In [14]:
import numpy as np
from PIL import Image

# ---------- Utilities ----------
def to_h(coords):
    """(N,2) -> (N,3) homogeneous"""
    return np.hstack([coords, np.ones((coords.shape[0], 1))])

def from_h(coords_h):
    """(N,3) -> (N,2) dehomogenize"""
    coords_h = coords_h / coords_h[:, [2]]
    return coords_h[:, :2]

def order_corners(pts):
    """
    Ensure pts are in consistent order: [top-left, top-right, bottom-right, bottom-left]
    pts: (4,2)
    """
    pts = np.asarray(pts, dtype=np.float64)
    c = pts.mean(axis=0)
    angles = np.arctan2(pts[:,1] - c[1], pts[:,0] - c[0])
    # Sort by angle, then rotate so that top-left (smallest x+y) comes first
    idx = np.argsort(angles)
    pts = pts[idx]
    tl_idx = np.argmin(pts.sum(axis=1))
    return np.roll(pts, -tl_idx, axis=0)

def segment_length(a, b):
    return np.linalg.norm(a - b)

def compute_homography_dlt(src_pts, dst_pts):
    """
    Compute H s.t. x' ~ H x  (DLT, unnormalized for simplicity)
    src_pts, dst_pts: (N,2), N>=4
    """
    src_pts = np.asarray(src_pts, dtype=np.float64)
    dst_pts = np.asarray(dst_pts, dtype=np.float64)
    N = src_pts.shape[0]
    A = []
    for i in range(N):
        x, y = src_pts[i]
        u, v = dst_pts[i]
        A.append([0, 0, 0, -x, -y, -1, v*x, v*y, v])
        A.append([x, y, 1, 0, 0, 0, -u*x, -u*y, -u])
    A = np.asarray(A, dtype=np.float64)
    # Solve Ah = 0 using SVD -> right singular vector corresponding to smallest singular value
    _, _, Vt = np.linalg.svd(A)
    h = Vt[-1, :]
    H = h.reshape(3, 3)
    return H

def bilinear_sample(im, xs, ys):
    """
    im: HxWxC float32/float64
    xs, ys: arrays of same shape giving floating src coords
    returns sampled pixels with shape xs.shape + (C,)
    """
    H, W = im.shape[:2]
    x0 = np.floor(xs).astype(np.int64)
    y0 = np.floor(ys).astype(np.int64)
    x1 = x0 + 1
    y1 = y0 + 1

    # clamp to valid range
    x0 = np.clip(x0, 0, W-1); x1 = np.clip(x1, 0, W-1)
    y0 = np.clip(y0, 0, H-1); y1 = np.clip(y1, 0, H-1)

    Ia = im[y0, x0]
    Ib = im[y0, x1]
    Ic = im[y1, x0]
    Id = im[y1, x1]

    wa = (x1 - xs) * (y1 - ys)
    wb = (xs - x0) * (y1 - ys)
    wc = (x1 - xs) * (ys - y0)
    wd = (xs - x0) * (ys - y0)

    return (Ia * wa[..., None] + Ib * wb[..., None] +
            Ic * wc[..., None] + Id * wd[..., None])

def rectify_single_image(image_path, src_pts, out_scale=1.0):
    """
    Rectify a single image of a planar surface.

    image_path : path to image
    src_pts    : 4 points picked on the plane in the image [(x,y), ...]
                 (order can be arbitrary; we’ll order them)
    out_scale  : scale factor for output size
    returns: rectified numpy image (HxWxC, uint8)
    """
    im = np.asarray(Image.open(image_path).convert('RGB'), dtype=np.float64) / 255.0

    # 1) Order corners consistently
    src = order_corners(np.array(src_pts, dtype=np.float64))

    # 2) Decide "ideal" rectangle size based on source geometry
    #    width = avg(top edge, bottom edge), height = avg(left edge, right edge)
    w_top    = segment_length(src[0], src[1])
    w_bottom = segment_length(src[3], src[2])
    h_left   = segment_length(src[0], src[3])
    h_right  = segment_length(src[1], src[2])

    W = int(out_scale * round((w_top + w_bottom) / 2.0))
    H = int(out_scale * round((h_left + h_right) / 2.0))
    W = max(W, 1); H = max(H, 1)

    # 3) Build destination rectangle (axis-aligned)
    dst = np.array([
        [0,   0],        # top-left
        [W-1, 0],        # top-right
        [W-1, H-1],      # bottom-right
        [0,   H-1],      # bottom-left
    ], dtype=np.float64)

    # 4) Compute H that maps src -> dst
    Hmat = compute_homography_dlt(src, dst)

    # 5) Inverse warp (for every output pixel, sample from input)
    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    dst_grid = to_h(np.stack([xx.ravel(), yy.ravel()], axis=1))
    src_grid_h = (np.linalg.inv(Hmat) @ dst_grid.T).T
    src_grid = from_h(src_grid_h)

    xs = src_grid[:, 0].reshape(H, W)
    ys = src_grid[:, 1].reshape(H, W)

    out = bilinear_sample(im, xs, ys)
    out = np.clip(out * 255.0, 0, 255).astype(np.uint8)
    return out


In [15]:
%matplotlib qt

In [18]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from PIL import Image, ImageOps

def pick_points(image_path, n, title):
    img = load_img_consistent(image_path)
    fig, ax = plt.subplots()
    ax.imshow(img)  
    # no rotation now
    ax.set_title(title)
    pts = np.array(plt.ginput(n, timeout=-1, show_clicks=True), dtype=float)
    plt.close(fig)
    return pts

def load_img_consistent(path, max_size=1200):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    
    w, h = img.size
    scale = max(w, h) / max_size
    print(scale)
    if scale > 1:
        new_size = (int(w / scale), int(h / scale))
        img = img.resize(new_size, Image.LANCZOS)
        print(f"Resized from {w}×{h} → {new_size}")
    
    return np.array(img)

im1_path = "/Users/jaitegchahal/cs180/jaitegchahal123.github.io/proj3/media/test.jpeg"
N = 4 
print(f"Select {N} points on LEFT image (in order).")
im1_pts = pick_points(im1_path, N, "LEFT image")

Select 4 points on LEFT image (in order).
1.0666666666666667
Resized from 1269×1280 → (1189, 1200)


In [19]:
rectified = rectify_single_image(im1_path, im1_pts, out_scale=1.0)
Image.fromarray(rectified).save("rectified_panel.png")
print("Saved -> rectified_panel.png")

Saved -> rectified_panel.png
